# 04a · Federated transfer, updated: elastic-net Cox, XGBoost, DeepHit and predictive checklists

Wrapper notebook for the second federated delivery (`rhino_data_v2`). All scoring logic lives in
`survival_common/federated_endpoint_transfer.py`; `04_federated_transfer.ipynb` still reads the first delivery
(`rhino_results`).

Four platinum-trained model families from the caia-project-compass `rhino_scripts` environments, each refit
**without Dana-Farber** (Fred Hutch, JHU and MSK), are scored on the full local DFCI ADT cohort. No DFCI patient
was in their training data, so this is an external validation on all three endpoints: **time to NEPC** and
**time to AVPC** (the fine-grained LLM endpoints) and **time to platinum** (the training endpoint, as a same-endpoint
reference).

| Family | v2 location (no-DFCI refit) | Scored as |
|---|---|---|
| Elastic-net Cox | `federated_elastic_net_cox_no_DFCI/` (landmarks 0/90/180 in one bundle) | linear predictor |
| XGBoost | `federated_xgboost*/lm_{0,90,180}/no_DFCI/` (one bundle per landmark) | boosted margin |
| Dynamic DeepHit | `federated_*deephit*/lm_{0,90,180}/no_DFCI/` (one bundle per landmark; landmark and dynamic arms) | CIF at each evaluation horizon |
| Predictive checklist | `federated_predictive_checklists/lm_{0,90,180}/no_DFCI/` (365- and 1825-day models) | raw item count, plus frozen probability and flag |

Bundles are found by file name anywhere under `RHINO_ROOT` in a `no_DFCI` / `*_no_DFCI` directory, so the
exact XGBoost and DeepHit folder names do not matter. Each family's per-landmark bundles are merged into one
(`merge_bundles`): the shared fields (format version, analysis label, time unit, library version) must agree, and
a landmark exported twice is an error. The pooled preprocessing in each bundle is applied as shipped and never refit.

**Metrics** for each endpoint × landmark × model (as in 04): mean AUC(t) over 1, 2, 3 and 5 years after the
landmark with a patient-bootstrap 95% CI (the figure metric), paired ΔAUC(t) and ΔC vs the age-only `baseline`,
Harrell's and Uno's C, per-horizon AUC(t), and Cox HR per SD of the score.
- The landmark XGBoost packages export no age-only model, so XGBoost is paired with the elastic-net Cox
  `baseline` (`baseline_bundle` in the metrics). An age-only model's discrimination depends only on how it ranks age,
  so this is the same comparator.
- DeepHit's dynamic arm is paired with the landmark arm's age baseline. DeepHit and checklist rows carry one row
  per evaluation (DeepHit) or model (checklist) horizon.

**Outputs**
- `OUTPUT_DIR/transfer_*.csv` (`rhino_data_v2/federated_transfer_adt_no_DFCI`): elastic-net Cox and XGBoost only,
  the federated Figure 5 inputs.
- `OUTPUT_DIR/all_models/`: all four families, for the checklist and all-models supplements. The Cox and XGBoost
  rows are the Figure 5 rows. DeepHit and the checklists are scored afterwards on the same frames and bootstrap
  resamples, so a failure in either cannot block the Figure 5 outputs.
- `OUTPUT_DIR/sequencing_anchor/` and the clinical comparison (`transfer_clinical_comparison.csv`,
  `transfer_clinical_cox_coefs.csv`, `transfer_subpop_risk_summary.csv`, `transfer_km_patients.csv`): Cox and
  XGBoost only, exactly as in 04 (Figure 5 b–h).

After this notebook, rerun `06_prep_figure_data.ipynb`, then `07_figures.R`, which read `rhino_data_v2` by default.

**Transferred model vs Gleason and TP53/RB1/PTEN** (federated Figure 5 b–d; unchanged from 04):
- **b · `gleason`**: patients with a Gleason score (ADT clock). Local Cox: **Gleason (continuous) + age**.
- **c · `trio`**: sequenced patients (sequencing clock). Local Cox: **TP53_ANY + RB1_ANY + PTEN_ANY + age**.
- **d · `trio_gleason`**: sequenced patients with a Gleason score (sequencing clock). Local Cox:
  **Gleason + TP53_ANY + RB1_ANY + PTEN_ANY + age**.

Each compares the transferred `both` score, the 5-fold out-of-fold local Cox, and a combined local Cox of the
transferred score + the clinical covariates without age. The somatic arms are anchored on the sequencing report date
(closest to ADT start); no somatic data is used on the ADT clock. Clinical data must be available strictly before the
landmark.

## Configuration

In [ ]:
import os, re, sys, json
from pathlib import Path

sys.path.insert(0, ".")
import compass_pipeline as cp

PROJECT_ROOT = cp.PROJECT_ROOT
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from survival_common.federated_inference import load_bundle, merge_bundles
from survival_common import federated_endpoint_transfer as fet
from survival_common import federated_scoring as fs

DATA_ROOT = cp._PROFILE_OUTPUT_ROOT

# >>> Edit: second federated delivery; transfer uses the bundles refit without DFCI (Fred Hutch + JHU + MSK) <<<
RHINO_ROOT = Path(os.environ.get("COMPASS_RHINO_ROOT", str(Path(DATA_ROOT) / "rhino_data_v2")))

DATA_PATH      = Path(DATA_ROOT) / "longitudinal_prediction_data_adt.csv"
OUTPUT_DIR     = RHINO_ROOT / "federated_transfer_adt_no_DFCI"
ALL_MODELS_DIR = OUTPUT_DIR / "all_models"
LAB_MAPPING    = fet.DEFAULT_LAB_MAPPING

ENDPOINTS   = ("nepc", "avpc", "platinum")
EXCLUSION   = "none"            # "none" = full ADT cohort; "pre_anchor_castrate" for noprecastrate bundles
LANDMARKS   = None              # None = every landmark present in the bundles
HORIZONS    = fet.DEFAULT_HORIZONS_DAYS   # AUC(t) horizons, days after the landmark
N_BOOTSTRAP = 200
SEED        = 0

# Figure 5 b-d: transferred labs score vs local Gleason / TP53-RB1-PTEN Cox
import federated_clinical_comparison as fcc
GLEASON_PATH          = fcc.sg.DEFAULT_GLEASON_PATH
SOMATIC_PATH          = fcc.sg.DEFAULT_SOMATIC_PATH
SOMATIC_MANIFEST_PATH = fcc.sg.DEFAULT_SOMATIC_MANIFEST_PATH
CLINICAL_CV_FOLDS     = 5     # out-of-fold folds for the local Cox
CLINICAL_MIN_EVENTS   = 5     # fewer events in a subpopulation -> count-only row

# Bundle and export-metrics file per family. The landmark packages write the same file name once per
# landmark (lm_0 / lm_90 / lm_180); the elastic-net Cox bundle holds every landmark.
BUNDLE_FILES = {
    "elastic_net_cox":      "cox_federated_elasticnet_model_adt.json",
    "xgboost_cox":          "xgboost_federated_model_adt.json",
    "dynamic_deephit":      "deephit_federated_model_adt.json",
    "predictive_checklist": "predictive_checklist_federated_model_adt.json",
}
METRIC_FILES = {
    "elastic_net_cox":      "cox_federated_elasticnet_metrics_adt.csv",
    "xgboost_cox":          "xgboost_federated_metrics_adt.csv",
    "dynamic_deephit":      "deephit_federated_metrics_adt.csv",
    "predictive_checklist": "predictive_checklist_metrics_adt.csv",
}
PRIMARY_FAMILIES = ("elastic_net_cox", "xgboost_cox")   # Figure 5 and the clinical comparison
BASELINE_DONOR   = {"xgboost_cox": "elastic_net_cox"}    # landmark XGBoost exports no age-only model

def is_no_dfci(path):
    return any(part == "no_DFCI" or part.endswith("_no_DFCI")
               for part in path.relative_to(RHINO_ROOT).parts)

def natural_key(path):   # lm_90 before lm_180
    return [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", str(path))]

BUNDLE_PATHS = {family: sorted((p for p in RHINO_ROOT.rglob(name)
                                if is_no_dfci(p) and OUTPUT_DIR not in p.parents), key=natural_key)
                for family, name in BUNDLE_FILES.items()}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for family, paths in BUNDLE_PATHS.items():
    print(f"{family}: {len(paths)} no-DFCI bundle(s)")
    for p in paths:
        print("  ", p.relative_to(RHINO_ROOT))

## Load and merge bundles

One bundle per family, keyed by its file stem. A family with no delivery yet is skipped with a warning; a landmark
delivered twice, or deliveries that disagree on format, label, time unit or library version, stop here.

In [ ]:
def load_family(family, paths):
    """Load one family's per-landmark bundles and merge them into one."""
    parts = [load_bundle(p) for p in paths]
    for p, b in zip(paths, parts):
        assert b.get("model_family") == family, f"{p}: model_family {b.get('model_family')!r}, expected {family!r}"
    return merge_bundles(parts, sources=[str(p) for p in paths])

bundles, SOURCES, rows = {}, {}, []
for family, paths in BUNDLE_PATHS.items():
    if not paths:
        print(f"WARNING: no no-DFCI {family} bundle ({BUNDLE_FILES[family]}) under {RHINO_ROOT}; skipped")
        continue
    stem = Path(BUNDLE_FILES[family]).stem
    b = bundles[stem] = load_family(family, paths)
    SOURCES[stem] = paths
    label = str(b.get("analysis_label") or "")
    if ("noprecastrate" in label) != (EXCLUSION == "pre_anchor_castrate"):
        print(f"WARNING: {stem} analysis_label={label!r} but EXCLUSION={EXCLUSION!r}")
    for m in b["models"]:
        rows.append({
            "bundle": stem, "model_family": family, "analysis_label": label,
            "trained_endpoint": b.get("endpoint", "platinum"),
            "landmark_days": m["landmark_days"], "config": m.get("config"),
            "training_arm": m.get("training_arm"), "model_horizon_days": m.get("horizon_days"),
        })

STEMS = {b["model_family"]: stem for stem, b in bundles.items()}
PRIMARY = [STEMS[f] for f in PRIMARY_FAMILIES if f in STEMS]
assert PRIMARY, f"no elastic-net Cox or XGBoost no-DFCI bundle under {RHINO_ROOT}"
BASELINE_FALLBACK = {STEMS[f]: STEMS[d] for f, d in BASELINE_DONOR.items() if f in STEMS and d in STEMS}
print("age-only baseline borrowed:", BASELINE_FALLBACK or "none")

def _values(s):
    return ", ".join(str(v) for v in sorted(s.dropna().unique())) or "–"

bundle_summary = pd.DataFrame(rows)
(bundle_summary.groupby(["model_family", "bundle"], sort=False)
    .agg(n_models=("config", "size"), landmarks=("landmark_days", _values), configs=("config", _values),
         arms=("training_arm", _values), model_horizons=("model_horizon_days", _values))
    .assign(sources=lambda d: [", ".join(str(p.parent.relative_to(RHINO_ROOT)) for p in SOURCES[s])
                               for s in d.index.get_level_values("bundle")]))

## Local cohort

In [ ]:
local = fet.read_local_frame(DATA_PATH, ENDPOINTS)
lab_mapping = fet.load_lab_mapping(LAB_MAPPING)

pt = local.drop_duplicates(fet.LOCAL_ID_COL)
print(f"{len(local):,} rows, {len(pt):,} patients from {DATA_PATH}")
pd.DataFrame({
    ep: {"events": int(pt[ind].fillna(0).astype(int).sum()),
         "dated events": int((pt[ind].fillna(0).astype(int).eq(1) & pt[dt].notna()).sum())}
    for ep, (ind, dt) in fet.ENDPOINT_SOURCES.items() if ep in ENDPOINTS
}).T

## Run transfer evaluation

The Figure 5 bundles (elastic-net Cox, XGBoost) are scored and written first. The bootstrap resamples depend only
on the seed and the (endpoint, landmark) frame, never on which bundles are scored, so the DeepHit and checklist run
below shares them and its rows can be appended to these.

In [ ]:
# Shared by every run so their bootstrap resamples and horizons match.
TRANSFER_KW = dict(
    endpoints=ENDPOINTS, lab_mapping=lab_mapping, exclusion=EXCLUSION,
    landmarks=LANDMARKS, horizons_days=HORIZONS,
    n_bootstrap=N_BOOTSTRAP, seed=SEED,
)

def combine(*results):
    """Row-bind transfer results scored on the same frames; prep_stats is per endpoint, so it is the first's."""
    return {k: pd.concat([r[k] for r in results], ignore_index=True)
               if isinstance(results[0][k], pd.DataFrame) else results[0][k]
            for k in results[0]}

def write_outputs(res, stems, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    for name in ("scores", "metrics", "auc_t", "coverage", "unit_check"):
        if not res[name].empty:
            res[name].to_csv(out_dir / f"transfer_{name}.csv", index=False)
    with open(out_dir / "transfer_run.json", "w") as fh:
        json.dump({
            "data": str(DATA_PATH), "rhino_root": str(RHINO_ROOT),
            "bundles": {s: {"paths": [str(p) for p in SOURCES[s]],
                            **{k: bundles[s].get(k) for k in (
                                "model_family", "analysis_label", "format_version",
                                "xgboost_version", "torch_version", "training_sites")}}
                        for s in stems},
            "baseline_fallback": {k: v for k, v in BASELINE_FALLBACK.items() if k in stems},
            "endpoints": list(ENDPOINTS), "exclusion": EXCLUSION,
            "landmarks": list(LANDMARKS) if LANDMARKS else None,
            "horizons_days": list(HORIZONS), "n_bootstrap": N_BOOTSTRAP, "seed": SEED,
            "prep_stats": res["prep_stats"],
        }, fh, indent=2, default=str)
    print(f"wrote outputs -> {out_dir}")

primary_bundles = {s: bundles[s] for s in PRIMARY}
res = fet.evaluate_transfer(local, primary_bundles, baseline_fallback=BASELINE_FALLBACK, **TRANSFER_KW)
write_outputs(res, PRIMARY, OUTPUT_DIR)   # Figure 5: one row per endpoint × bundle × landmark × config

In [ ]:
# DeepHit and checklists (each paired with its own age-only model), appended to the Figure 5 rows for the
# checklist and all-models supplements.
EXTRA = [s for s in bundles if s not in PRIMARY]
if EXTRA:
    extra_res = fet.evaluate_transfer(local, {s: bundles[s] for s in EXTRA}, **TRANSFER_KW)
    all_res = combine(res, extra_res)
else:
    print("no DeepHit or checklist bundle; all_models/ holds the Figure 5 bundles only")
    all_res = res
write_outputs(all_res, list(bundles), ALL_MODELS_DIR)

In [ ]:
# Trio arm (Figure 5 c) on the sequencing clock: the Cox/XGBoost bundles re-scored with each sequenced patient's
# sequencing report date (closest to ADT start) as the anchor, so labs come from before sequencing + landmark and
# outcomes run from sequencing + landmark.
clin_gleason, clin_somatic, clin_somatic_features = fcc.load_clinical_sources(
    GLEASON_PATH, SOMATIC_PATH, SOMATIC_MANIFEST_PATH)
specimen = fcc.select_specimen(local, clin_somatic, clin_somatic_features)
local_seq = local
if EXCLUSION != "none":
    # Judge the exclusion on the ADT clock, then re-anchor the patients it keeps.
    kept, _ = fs.apply_exclusion(fet.to_federated_long(local, "platinum", lab_mapping)[0], EXCLUSION)
    local_seq = local[pd.to_numeric(local[fet.LOCAL_ID_COL], errors="coerce").isin(kept[fs.ID_COL].unique())]
local_seq = fcc.sequencing_anchored_local(local_seq, specimen)
seq_offset = specimen[fcc.sg.SEQUENCING_TO_ADT_DAYS]
print(f"{local_seq[fet.LOCAL_ID_COL].nunique():,} sequenced patients; sequencing − ADT start median "
      f"{seq_offset.median():.0f}d (IQR {seq_offset.quantile(.25):.0f} to {seq_offset.quantile(.75):.0f}d), "
      f"{int(seq_offset.lt(0).sum()):,} sequenced before ADT")

SEQ_OUTPUT_DIR = OUTPUT_DIR / "sequencing_anchor"
res_seq = fet.evaluate_transfer(local_seq, primary_bundles, baseline_fallback=BASELINE_FALLBACK,
                                **{**TRANSFER_KW, "exclusion": "none"})
write_outputs(res_seq, PRIMARY, SEQ_OUTPUT_DIR)

## Sanity checks: feature coverage and units

Check these before reading any metric. A covariate the local frame never produces is imputed with the
pooled mean for every patient, which carries no signal. A local mean more than 3× off the pooled mean
usually means a unit mismatch.

The pooled impute means come from Fred Hutch, JHU and MSK only.

In [ ]:
cov = all_res["coverage"]
display(cov.head())
num = cov.select_dtypes("number").columns.difference(["landmark_days", "model_horizon_days"])
cov.groupby(["bundle", "landmark_days", "config", "training_arm", "model_horizon_days"],
            dropna=False)[list(num)].mean().round(3)

In [ ]:
uc = all_res["unit_check"]
flagged = uc[uc["unit_flag"]].sort_values("mean_ratio", ascending=False)
print(f"{flagged['covariate'].nunique()} covariate(s) flagged (>3x from pooled mean)")
flagged.drop_duplicates("covariate")

## Discrimination

One table per family. DeepHit rows are per training arm and evaluation horizon (the CIF at that horizon, so mean
AUC(t) is the AUC at that horizon); checklist rows are per model horizon.

In [ ]:
def fmt_ci(est, lo, hi, d=3):
    if not np.isfinite(est):
        return ""
    return f"{est:.{d}f} [{lo:.{d}f}, {hi:.{d}f}]" if np.isfinite(lo) else f"{est:.{d}f}"

KEY_COLS = ["endpoint", "landmark_days", "model_family", "bundle", "training_arm",
            "model_horizon_days", "evaluation_horizon_days"]

def summarize_metrics(metrics):
    """Add formatted CI columns; return (metrics, table of the non-baseline configs)."""
    m = metrics.copy()
    for c in ("c_index_lo", "c_index_hi", "delta_c_vs_baseline", "delta_c_lo", "delta_c_hi",
              "mean_auc_lo", "mean_auc_hi", "delta_auc_vs_baseline", "delta_auc_lo", "delta_auc_hi",
              "baseline_bundle"):
        if c not in m:
            m[c] = np.nan
    m["C (95% CI)"] = [fmt_ci(r.c_index, r.c_index_lo, r.c_index_hi) for r in m.itertuples()]
    m["ΔC vs age (95% CI)"] = [fmt_ci(r.delta_c_vs_baseline, r.delta_c_lo, r.delta_c_hi) for r in m.itertuples()]
    m["Mean AUC(t) (95% CI)"] = [fmt_ci(r.mean_auc, r.mean_auc_lo, r.mean_auc_hi) for r in m.itertuples()]
    m["ΔAUC vs age (95% CI)"] = [fmt_ci(r.delta_auc_vs_baseline, r.delta_auc_lo, r.delta_auc_hi)
                                 for r in m.itertuples()]
    m["HR/SD (95% CI)"] = [fmt_ci(r.hr_per_sd, r.hr_per_sd_lo, r.hr_per_sd_hi, 2) for r in m.itertuples()]
    summary = m[m["config"] != "baseline"][[
        *KEY_COLS, "n_patients", "n_events",
        "Mean AUC(t) (95% CI)", "ΔAUC vs age (95% CI)", "C (95% CI)", "ΔC vs age (95% CI)",
        "HR/SD (95% CI)", "uno_c", "baseline_bundle",
    ]].round({"uno_c": 3})
    return m, summary.sort_values(KEY_COLS, na_position="first").reset_index(drop=True)

m, summary = summarize_metrics(all_res["metrics"])
for family, table in summary.groupby("model_family", sort=False):
    print(family)
    display(table.dropna(axis=1, how="all").reset_index(drop=True))

In [ ]:
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]          # fixed categorical order

def plot_mean_auc(metrics, title, path):
    """Mean AUC(t) (95% bootstrap CI) by landmark; one panel per endpoint, full vs age-only."""
    plot = metrics.dropna(subset=["mean_auc"])
    panels = [ep for ep in ENDPOINTS if ep in set(plot["endpoint"])]
    stems = sorted(plot["bundle"].unique())
    color = {s: COLORS[i % len(COLORS)] for i, s in enumerate(stems)}
    lms = sorted(plot["landmark_days"].unique())
    width = 0.8 / max(1, 2 * len(stems))

    fig, axes = plt.subplots(1, len(panels), figsize=(4.2 * len(panels), 3.6), sharey=True, squeeze=False)
    for ax, ep in zip(axes[0], panels):
        d = plot[plot["endpoint"] == ep]
        k = 0
        for s in stems:
            for cfg, mk, fill in (("both", "o", True), ("baseline", "s", False)):
                r = d[(d["bundle"] == s) & (d["config"] == cfg)].set_index("landmark_days").reindex(lms)
                x = np.arange(len(lms)) + (k - (2 * len(stems) - 1) / 2) * width
                k += 1
                if r["mean_auc"].isna().all():
                    continue
                yerr = np.vstack([r["mean_auc"] - r["mean_auc_lo"], r["mean_auc_hi"] - r["mean_auc"]])
                ax.errorbar(x, r["mean_auc"], yerr=yerr, fmt=mk, ms=7, lw=1.5, capsize=0,
                            color=color[s], mfc=color[s] if fill else "white", mew=1.5,
                            label=f"{s} · {'full' if cfg == 'both' else 'age-only'}")
        ax.axhline(0.5, color="#8a8a86", lw=1, ls="--", zorder=0)
        ax.set_xticks(range(len(lms)), [f"{lm}d" for lm in lms])
        n_ev = d.groupby("landmark_days")["n_events"].first().reindex(lms)
        ax.set_title(f"{ep.upper()}  (events: {', '.join(str(int(v)) for v in n_ev.fillna(0))})", fontsize=10)
        ax.set_xlabel("landmark")
        ax.grid(axis="y", color="#e6e5df", lw=0.8)
        ax.spines[["top", "right"]].set_visible(False)
    axes[0][0].set_ylabel("Mean AUC(t)")
    axes[0][-1].legend(loc="upper left", bbox_to_anchor=(1.02, 1), frameon=False, fontsize=8)
    fig.suptitle(title, fontsize=11)
    fig.tight_layout()
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.show()

# XGBoost has no age-only point of its own: its ΔAUC is against the elastic-net Cox age-only model.
plot_mean_auc(res["metrics"], "Platinum-trained federated models without DFCI on local endpoints (external)",
              OUTPUT_DIR / "transfer_mean_auc.png")

### All four families: AUC(t) by horizon

Each family at the horizons it is scored on: Cox and XGBoost risk scores at every horizon, DeepHit's CIF at each
horizon, and each checklist at its own model horizon (unconnected: each point is a separate model). Colors follow
the family; the DeepHit arms share a color and differ by line style.

In [ ]:
SERIES = {   # (family, config, arm) -> (label, color, line style, marker)
    ("elastic_net_cox", "both", None):         ("Elastic-net Cox", COLORS[0], "-", "o"),
    ("xgboost_cox", "both", None):             ("XGBoost", COLORS[1], "-", "o"),
    ("dynamic_deephit", "both", "dynamic"):    ("DeepHit · dynamic arm", COLORS[2], "-", "o"),
    ("dynamic_deephit", "both", "landmark"):   ("DeepHit · landmark arm", COLORS[2], "--", "s"),
    ("predictive_checklist", "both", None):    ("Predictive checklist", COLORS[3], "none", "D"),  # one model per point
    ("elastic_net_cox", "baseline", None):     ("Age only", "#8a8a86", ":", "^"),
}

def plot_auc_t_all(auc_t, path):
    """AUC(t) vs years after the landmark; rows = endpoints, columns = landmarks."""
    family = {s: b["model_family"] for s, b in bundles.items()}
    d = auc_t.assign(model_family=auc_t["bundle"].map(family), years=auc_t["horizon_days"] / 365.25)
    eps = [ep for ep in ENDPOINTS if ep in set(d["endpoint"])]
    lms = sorted(d["landmark_days"].unique())
    fig, axes = plt.subplots(len(eps), len(lms), figsize=(3.4 * len(lms), 2.8 * len(eps)),
                             sharex=True, sharey=True, squeeze=False)
    handles = {}
    for i, ep in enumerate(eps):
        for j, lm in enumerate(lms):
            ax = axes[i][j]
            panel = d[(d["endpoint"] == ep) & (d["landmark_days"] == lm)]
            for (fam, cfg, arm), (label, color, ls, mk) in SERIES.items():
                r = panel[(panel["model_family"] == fam) & (panel["config"] == cfg)]
                r = r[r["training_arm"].isna()] if arm is None else r[r["training_arm"] == arm]
                if r.empty:
                    continue
                r = r.sort_values("horizon_days")
                (line,) = ax.plot(r["years"], r["auc"], ls=ls, marker=mk, ms=8, lw=2, color=color,
                                  mfc=color if cfg == "both" else "white", mew=1.5)
                handles.setdefault(label, line)
            ax.axhline(0.5, color="#8a8a86", lw=1, ls="--", zorder=0)
            ax.grid(axis="y", color="#e6e5df", lw=0.8)
            ax.spines[["top", "right"]].set_visible(False)
            if i == 0:
                ax.set_title(f"landmark {lm}d", fontsize=10)
            if j == 0:
                ax.set_ylabel(f"{ep.upper()}\nAUC(t)")
            if i == len(eps) - 1:
                ax.set_xlabel("years after landmark")
    fig.legend(handles.values(), handles.keys(), loc="upper left", bbox_to_anchor=(1.0, 0.95),
               frameon=False, fontsize=8)
    fig.suptitle("Federated models without DFCI, all families: AUC(t) on local endpoints", fontsize=11)
    fig.tight_layout()
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.show()

if all_res["auc_t"].empty:
    print("no estimable AUC(t) horizons")
else:
    plot_auc_t_all(all_res["auc_t"], ALL_MODELS_DIR / "transfer_auc_t_all_models.png")

In [ ]:
# DeepHit: landmark vs dynamic arm (AUC at each evaluation horizon, of the CIF at that horizon)
dh = m[m["model_family"] == "dynamic_deephit"]
if dh.empty:
    print("no DeepHit bundle scored")
else:
    display(dh.pivot_table(index=["endpoint", "landmark_days", "evaluation_horizon_days"],
                           columns=["config", "training_arm"], values="mean_auc").round(3))

### Local platinum vs the federated held-out test

The federated numbers come from each bundle's own pooled 80/20 split of the non-DFCI sites, read from the export
metrics beside every source bundle (concatenated over the per-landmark deliveries). The local platinum numbers are
external estimates on DFCI patients.
- Cox, XGBoost and DeepHit: federated `test_c_index` and `test_mean_auc_t`. For DeepHit, whose federated risk is
  CIF_total, the local C is the CIF at the longest evaluation horizon and the local AUC is the mean of the
  per-horizon AUC(t).
- Checklists: federated `auc` on `dataset == "test"` at the model horizon (no C-index is exported).

In [ ]:
MATCH_KEYS = ["bundle", "model_family", "landmark_days", "config", "training_arm", "model_horizon_days"]

def _keyed(d):
    return d.assign(training_arm=d["training_arm"].fillna("none"),
                    model_horizon_days=pd.to_numeric(d["model_horizon_days"]).fillna(-1).astype(int),
                    landmark_days=d["landmark_days"].astype(int))

def read_federated_metrics(stem):
    """Export metrics beside each source bundle of a merged family, concatenated."""
    family = bundles[stem]["model_family"]
    frames = [pd.read_csv(p.with_name(METRIC_FILES[family]))
              for p in SOURCES[stem] if p.with_name(METRIC_FILES[family]).exists()]
    if not frames:
        return pd.DataFrame(columns=MATCH_KEYS)
    fed = pd.concat(frames, ignore_index=True)
    if "cohort" in fed:
        fed = fed[fed["cohort"].astype(str).str.lower() == "all"]
    if family == "predictive_checklist":
        fed = fed[fed["dataset"] == "test"].rename(columns={
            "horizon_days": "model_horizon_days", "n": "n_test", "n_cases": "n_events_test", "auc": "test_auc"})
        fed["test_c_index"] = np.nan
    else:
        fed = fed.rename(columns={"test_mean_auc_t": "test_auc"})
        fed["model_horizon_days"] = np.nan
    if "training_arm" not in fed:
        fed["training_arm"] = "landmark" if family == "dynamic_deephit" else np.nan
    fed = _keyed(fed.assign(bundle=stem, model_family=family))
    return fed[MATCH_KEYS + ["n_test", "n_events_test", "test_c_index", "test_auc"]].drop_duplicates(MATCH_KEYS)

def local_platinum(metrics):
    loc = _keyed(metrics[metrics["endpoint"] == "platinum"])
    longest = loc.groupby(MATCH_KEYS)["evaluation_horizon_days"].transform("max")
    last = loc[loc["evaluation_horizon_days"].isna() | loc["evaluation_horizon_days"].eq(longest)]
    auc = loc.groupby(MATCH_KEYS, as_index=False)["mean_auc"].mean()
    return last[MATCH_KEYS + ["n_patients", "n_events", "c_index"]].merge(auc, on=MATCH_KEYS)

def federated_vs_local(metrics):
    fed = pd.concat([read_federated_metrics(s) for s in bundles], ignore_index=True)
    return (fed.merge(local_platinum(metrics), on=MATCH_KEYS, how="outer")
        .rename(columns={"test_c_index": "federated_test_C", "test_auc": "federated_test_AUC",
                         "c_index": "local_platinum_C", "mean_auc": "local_platinum_AUC",
                         "n_patients": "local_n", "n_events": "local_events"})
        .replace({"training_arm": {"none": ""}, "model_horizon_days": {-1: np.nan}})
        .sort_values(MATCH_KEYS).round(3).reset_index(drop=True))

federated_vs_local(all_res["metrics"])

## Time-dependent AUC

In [ ]:
auc = all_res["auc_t"]
if auc.empty:
    print("no estimable AUC(t) horizons")
else:
    display(auc.assign(training_arm=lambda d: d["training_arm"].fillna("none"),
                       model_horizon_days=lambda d: d["model_horizon_days"].fillna(-1).astype(int))
            .pivot_table(index=["endpoint", "landmark_days", "bundle", "config", "training_arm",
                                "model_horizon_days"], columns="horizon_days", values="auc").round(3))

## Risk-score distributions

The feature frame does not depend on the endpoint, so the score distribution is the same for every
endpoint. The check that matters is whether local scores fall inside the range the pooled model was
fitted on. DeepHit scores are CIFs and checklist scores are item counts, so compare within a bundle only.

In [ ]:
sc = all_res["scores"]
sc0 = sc[(sc["endpoint"] == ENDPOINTS[0]) & (sc["config"] == "both")]
sc0.groupby(["bundle", "landmark_days", "training_arm", "model_horizon_days", "evaluation_horizon_days"],
            dropna=False)["risk_score"].describe().round(3)

## Transferred model vs Gleason and TP53/RB1/PTEN

These are the inputs for federated Figure 5 b–d, on the elastic-net Cox and XGBoost bundles only. Each
subpopulation is defined per landmark:
- **gleason** (ADT clock): patients with a Gleason score available before TREATMENT_ANCHOR_DATE + landmark. The
  score closest to the landmark is used, and it counts as available from the later of the score date and the date
  of the note that reported it.
- **trio** (sequencing clock): sequenced patients, with the landmark measured from the sequencing date. The
  sequencing date is the report date of the test reported closest to ADT start (same-day reports combined), and
  that specimen is the anchor at every landmark.
  Each gene's `_ANY` indicator is 1 for an SNV, deletion, amplification or SV. Age is age at sequencing.
- **trio_gleason** (sequencing clock): the trio patients who also have a Gleason score available before
  sequencing + landmark.

The transferred scores are the frozen bundles, never refit: `res` (ADT clock) for the Gleason arm and
`res_seq` (sequencing clock) for the trio and trio + Gleason arms. The local and combined (transferred score +
clinical) Cox models are fitted on these patients, so their scores are out-of-fold (stratified on the event). The
apparent C and mean AUC(t) are in the CSV for reference only.

In [ ]:
lms = sorted(res["scores"]["landmark_days"].unique())
# Gleason arm on the ADT clock (Gleason and age only: no somatic data on the ADT clock); trio and
# trio + Gleason arms on the sequencing clock (trio taken from the anchoring specimen, Gleason available
# before sequencing + landmark).
clinical = fcc.build_clinical_features(local, lms, gleason=clin_gleason)
clinical_seq = fcc.build_clinical_features(
    local_seq, lms, gleason=clin_gleason, specimen=specimen, somatic_features=clin_somatic_features)

def membership(scores, clin):
    """Membership among patients actually scored at each landmark (the landmark frame applies the incident gate)."""
    scored_pt = scores.drop_duplicates(["endpoint", "landmark_days", fs.ID_COL])
    member = scored_pt.merge(clin, on=[fs.ID_COL, "landmark_days"], how="left")
    member["any_trio_altered"] = member[list(fcc.TRIO_FEATURES)].eq(1).any(axis=1)
    member["sequenced_with_gleason"] = member[fcc.HAS_SEQUENCING].notna() & member[fcc.GLEASON_FEATURE].notna()
    return member.groupby(["endpoint", "landmark_days"]).agg(
        n_scored=(fs.ID_COL, "size"),
        with_gleason=(fcc.GLEASON_FEATURE, "count"),
        sequenced=(fcc.HAS_SEQUENCING, "count"),
        sequenced_with_gleason=("sequenced_with_gleason", "sum"),
        any_trio_altered=("any_trio_altered", "sum"),
    )

(membership(res["scores"], clinical)[["n_scored", "with_gleason"]]
    .join(membership(res_seq["scores"], clinical_seq)[["sequenced", "sequenced_with_gleason", "any_trio_altered"]]
          .add_suffix(" (sequencing clock)")))

In [ ]:
def clinical_table(comp):
    """One row per endpoint × landmark × subpopulation × bundle: mean AUC(t) of each model and the paired ΔAUC(t)."""
    c = comp.assign(
        AUC=[fmt_ci(r.mean_auc, r.mean_auc_lo, r.mean_auc_hi) for r in comp.itertuples()],
        dAUC=[fmt_ci(r.delta_auc_vs_transferred, r.delta_auc_lo, r.delta_auc_hi) for r in comp.itertuples()],
        dC=[fmt_ci(r.delta_c_vs_transferred, r.delta_lo, r.delta_hi) for r in comp.itertuples()],
        dAUC_clin=[fmt_ci(r.delta_auc_vs_clinical, r.delta_auc_vs_clinical_lo, r.delta_auc_vs_clinical_hi)
                   for r in comp.itertuples()],
    )
    keys = ["endpoint", "landmark_days", "subpopulation", "bundle", "n_scored", "n_scored_events"]
    wide = c.pivot_table(index=keys, columns="model", values="AUC", aggfunc="first")
    wide = wide.reindex(columns=list(fcc.MODELS))
    wide.columns = ["AUC(t) labs (transferred)", "AUC(t) age (transferred)", "AUC(t) local Cox (OOF)",
                    "AUC(t) labs + local Cox (OOF)"]
    cox = c[c["model"] == fcc.CLINICAL_COX].set_index(keys)
    combined = c[c["model"] == fcc.COMBINED_COX].set_index(keys)
    wide["ΔAUC(t) Cox − labs (95% CI)"] = cox["dAUC"]
    wide["ΔAUC(t) labs + Cox − labs (95% CI)"] = combined["dAUC"]
    wide["ΔAUC(t) labs + Cox − Cox (95% CI)"] = combined["dAUC_clin"]
    wide["ΔC Cox − labs (95% CI)"] = cox["dC"]
    return wide.reset_index()

def subpop_summary(scores_adt, scores_seq):
    """Score strata: full cohort and Gleason on the ADT clock, sequencing strata on the sequencing clock."""
    return pd.concat([
        fcc.subpopulation_score_summary(scores_adt, clinical, sections=("cohort", "gleason"))
            .assign(time_origin="ADT start"),
        fcc.subpopulation_score_summary(scores_seq, clinical_seq, sections=("sequencing",))
            .assign(time_origin="sequencing date"),
    ], ignore_index=True)

comp, coefs = fcc.compare_by_subpopulation(
    {"gleason": (res["scores"], clinical), "trio": (res_seq["scores"], clinical_seq),
     "trio_gleason": (res_seq["scores"], clinical_seq)},
    n_folds=CLINICAL_CV_FOLDS, n_bootstrap=N_BOOTSTRAP,
    seed=SEED, min_events=CLINICAL_MIN_EVENTS, horizons_days=HORIZONS)
subpop = subpop_summary(res["scores"], res_seq["scores"])
# Patient-level labs scores + strata for the Figure 5 e–h KM panels (score groups are assigned in R).
km = fcc.km_patients({"adt": (res["scores"], clinical), "sequencing": (res_seq["scores"], clinical_seq)})
fcc.write_outputs(comp, coefs, subpop, OUTPUT_DIR, km=km)

fit_issues = comp[(comp["n_failed_folds"] > 0) | (comp["n_convergence_warnings"] > 0) | comp["note"].ne("")]
if len(fit_issues):
    display(fit_issues.drop_duplicates(["endpoint", "landmark_days", "subpopulation"])[
        ["endpoint", "landmark_days", "subpopulation", "n_patients", "n_events", "n_folds",
         "n_failed_folds", "n_convergence_warnings", "note"]])
clinical_table(comp)

In [ ]:
# Local Cox fitted on the whole subpopulation (apparent fit; HR per unit, age per year)
display(coefs.assign(**{"HR (95% CI)": [fmt_ci(r.hr, r.hr_lo, r.hr_hi, 2) for r in coefs.itertuples()]})
        .pivot_table(index=["subpopulation", "covariate"], columns=["endpoint", "landmark_days"],
                     values="HR (95% CI)", aggfunc="first"))

# Transferred labs score inside the subpopulations (pooled over endpoints; the score does not depend on them)
subpop.pivot_table(index=["landmark_days", "stratum"], columns="bundle", values="median", sort=False).round(3)

In [ ]:
fcc.plot_clinical_comparison(comp, OUTPUT_DIR / "transfer_clinical_comparison.png", ENDPOINTS)